# Determine file substructure.
## Credentials/
### -Application/
#### --Github/
##### ---Docker/
###### ----fictional-garbonzo/
###### -----private-a
###### -----public-b
###### ----other-repo/
###### -----private-a
###### -----public-b
##### ---Non-Env repo/
###### ----public
###### ----private
###### ----private API keys
###### ----public API keys
#### --Docker/
##### ---fictonal-garbanzo/
###### ----private-a
###### ----public-a
###### ----private-a
###### ----public-b
###### ----private-a
###### ----public-b
#### --CLI/
##### ---some-utility
###### ----username-password combo
##### ---network stuff
###### ----username-password combo
###### ----identifiable info {ip addresses, passwords, etc.
#### --Other/
##### ---other/


In [23]:
import nbformat
import pathlib
import types
import sys
import os
import json
import hashlib
import logging
from datetime import datetime
import random
import string
import re

# def load_notebook(nb_path):
#     path = pathlib.Path(nb_path).resolve()
#     nb   = nbformat.read(path, as_version=4)
#     mod  = types.ModuleType(path.stem)
#     mod.__file__ = str(path)
#     exec(compile(
#         "\n".join(c.source for c in nb.cells if c.cell_type == "code"),
#         str(path), "exec"
#     ), mod.__dict__)
#     sys.modules[path.stem] = mod
#     return mod
import traceback

try:
    ledger_mod = load_notebook(ROOT / "Ledger" / "Blockchain.ipynb")
except Exception:
    traceback.print_exc()

def load_notebook(nb_path):
    path = pathlib.Path(nb_path).resolve()
    nb   = nbformat.read(path, as_version=4)
    mod  = types.ModuleType(path.stem)
    mod.__file__ = str(path)
    
    # pre-inject common names the notebook assumes are already defined
    mod.__dict__.update({
        "Path":      pathlib.Path,
        "pathlib":   pathlib,
        "os":        os,
        "sys":       sys,
        "json":      json,
        "hashlib":   hashlib,
        "logging":   logging,
        "datetime":  datetime,
        "random":    random,
        "string":    string,
        "re":        re,
    })
    
    exec(compile(
        "\n".join(c.source for c in nb.cells if c.cell_type == "code"),
        str(path), "exec"
    ), mod.__dict__)
    sys.modules[path.stem] = mod
    return mod

# usage
ledger = load_notebook("../Ledger/Blockchain.ipynb")
Ledger = ledger.Ledger 

INFO: No existing ledger — initialising genesis block.
INFO: Block #1 committed — hash: ee9b301785a67348...
INFO: Block #2 committed — hash: a476865d99e92732...
INFO: Ledger loaded — 3 block(s), chain valid.
INFO: No existing ledger — initialising genesis block.
INFO: Block #1 committed — hash: 6e9e04c37f002331...
INFO: Block #2 committed — hash: 125e5faf80246b77...
INFO: Ledger loaded — 3 block(s), chain valid.


Genesis length: 1
Chain valid ✔
Tamper detection ✔
Ledger tests passed ✔
Genesis length: 1
Chain valid ✔
Tamper detection ✔
Ledger tests passed ✔


In [ ]:

%run "../Ledger/Blockchain.ipynb"
%run "../Module-Developer/Save-File.ipynb"
# every class/function defined in those notebooks
# is now available in this notebook's namespace

In [24]:



def format_credential(
    group_sizes: list[int]     = (5, 4, 5),   # characters per group
    delimiter:   str           = "-",          # separator between groups
    upper_count: int           = 1,            # uppercase letters per group
    digit_count: int           = 0,            # digits per group
    symbol_count:int           = 0,            # symbols per group
    upper_position: str        = "first",      # "first" | "last" | "random" | "spread"
) -> str:
    """
    Builds a format-pattern string from structural parameters.

    Placeholders:
        x  lowercase letter
        X  uppercase letter
        #  digit  (0-9)
        @  ASCII symbol
        -  delimiter (passed through verbatim)

    Args:
        group_sizes:    Number of characters in each group.
                        e.g. (5, 4, 5) → xxxxx-xxxx-xxxxx
        delimiter:      Character(s) placed between groups.
                        e.g. "-", ":", "::" , " "
        upper_count:    How many characters per group are uppercase (X).
        digit_count:    How many characters per group are digits (#).
        symbol_count:   How many characters per group are symbols (@).
        upper_position: Where to place uppercase characters within a group.
                        "first"  → X always leads   e.g. Xxxxx
                        "last"   → X always trails   e.g. xxxxX
                        "random" → X at random slots  e.g. xxXxx
                        "spread" → X evenly distributed e.g. xXxxXx

    Returns:
        A pattern string such as "Xxxxx-Xxxx-Xxxxx"

    Raises:
        ValueError: if reserved placeholder characters are used as the delimiter,
                    or if slot counts exceed the group size.
    """
    RESERVED = set("xX#@")
    if any(c in RESERVED for c in delimiter):
        raise ValueError(f"Delimiter '{delimiter}' must not contain x X # @")

    def _build_group(size: int) -> list[str]:
        total_special = upper_count + digit_count + symbol_count
        if total_special > size:
            raise ValueError(
                f"upper_count({upper_count}) + digit_count({digit_count}) + "
                f"symbol_count({symbol_count}) = {total_special} exceeds "
                f"group size {size}."
            )

        # Start with all lowercase slots
        slots = ["x"] * size

        # ── Place digits and symbols first (positionally agnostic) ──────────
        special_indices = random.sample(range(size), total_special)
        idx = iter(special_indices)

        digit_slots  = [next(idx) for _ in range(digit_count)]
        symbol_slots = [next(idx) for _ in range(symbol_count)]
        upper_slots  = [next(idx) for _ in range(upper_count)]   # remainder

        for i in digit_slots:
            slots[i] = "#"
        for i in symbol_slots:
            slots[i] = "@"

        # ── Place uppercase according to upper_position ──────────────────────
        lower_indices = [i for i in range(size) if slots[i] == "x"]

        match upper_position:
            case "first":
                chosen = sorted(lower_indices)[:upper_count]
            case "last":
                chosen = sorted(lower_indices)[-upper_count:]
            case "random":
                chosen = random.sample(lower_indices, upper_count)
            case "spread":
                # Evenly space across available lowercase positions
                if upper_count > len(lower_indices):
                    chosen = lower_indices
                else:
                    step   = len(lower_indices) / upper_count
                    chosen = [lower_indices[round(i * step)] for i in range(upper_count)]
            case _:
                raise ValueError(
                    f"upper_position must be 'first', 'last', 'random', or 'spread'. "
                    f"Got '{upper_position}'."
                )

        for i in chosen:
            slots[i] = "X"

        return slots

    groups  = [_build_group(size) for size in group_sizes]
    pattern = delimiter.join("".join(g) for g in groups)
    return pattern

In [25]:
# Default — mirrors original behaviour
format_credential()
# → "Xxxxx-Xxxx-Xxxxx"

# Two uppercase per group, spread out
format_credential(upper_count=2, upper_position="spread")
# → "XxxXx-XxxX-XxxXx"

# PIN-style: digits only, no letters
format_credential(group_sizes=(3, 3), upper_count=0, digit_count=3)
# → "###-###"

# Complex credential with symbols
format_credential(
    group_sizes=(4, 4, 4),
    delimiter=":",
    upper_count=1,
    digit_count=2,
    symbol_count=1,
    upper_position="last",
)
# → "##x@:x##X:@x#X"  (positions vary, pattern is deterministic in structure)

# Long passphrase-style, space-delimited
format_credential(group_sizes=(6, 6, 6, 6), delimiter=" ", upper_count=2)
# → "XxXxxx XxXxxx XxXxxx XxXxxx"

'XXxxxx XXxxxx XXxxxx XXxxxx'

In [26]:


# def format_credential():
#     """
#     Returns a format pattern string using placeholders:
#       x = lowercase letter
#       X = uppercase letter
#       # = digit (0-9)
#       @ = special ASCII symbol
#     Delimiters like - are passed through as-is.
#     """
#     return "xxxxx-Xxx##-xxxxx"   # Example: "abcde-Fgh42-ijklm"


def create_credential(length=None, 
                      char_count=None, 
                      include_special_char=True, 
                      formatter=None):
    """
    Generates a credential string based on a format pattern.

    Args:
        length:              Total expected length (optional, used for validation).
        char_count:          Number of non-delimiter characters expected (optional).
        include_special_char: If False, '@' placeholders are replaced with digits.

    Returns:
        A formatted credential string.
    """
    LOWERCASE  = string.ascii_lowercase          # a-z
    UPPERCASE  = string.ascii_uppercase          # A-Z
    DIGITS     = string.digits                   # 0-9
    SYMBOLS    = "!@#$%^&*()_+=-"               # standard ASCII symbols

    pattern = formatter
    result  = []

    for ch in pattern:
        match ch:
            case 'x':
                result.append(random.choice(LOWERCASE))
            case 'X':
                result.append(random.choice(UPPERCASE))
            case '#':
                result.append(random.choice(DIGITS))
            case '@':
                # Honour the include_special_char flag
                result.append(
                    random.choice(SYMBOLS) if include_special_char
                    else random.choice(DIGITS)
                )
            case _:
                # Delimiters (-  :  .  space, etc.) pass through unchanged
                result.append(ch)

    credential = "".join(result)

    # ── Optional validation ──────────────────────────────────────────────────
    if length is not None and len(credential) != length:
        raise ValueError(
            f"Generated credential length {len(credential)} "
            f"does not match requested length {length}."
        )

    non_delimiters = re.sub(r"[-:.\s]", "", credential)
    if char_count is not None and len(non_delimiters) != char_count:
        raise ValueError(
            f"Non-delimiter character count {len(non_delimiters)} "
            f"does not match requested char_count {char_count}."
        )

    return credential


# # ── Quick demo ───────────────────────────────────────────────────────────────
# if __name__ == "__main__":
#     for _ in range(5):
#         print(create_credential(include_special_char=True, 
#                                 formatter = format_credential(
#                                                   group_sizes    =  (2, 2, 2),   # characters per group
#                                                   delimiter      = ":",          # separator between groups
#                                                   upper_count    = 1,            # uppercase letters per group
#                                                   digit_count    = 0,            # digits per group
#                                                   symbol_count   = 0,            # symbols per group
#                                                   upper_position = "first",      # "first" | "last" | "random" | "spread")
#                 )
#             )
#         )
        

In [27]:


def save_credential(
    credential: str,
    filename:   str  = "credential",
    directory:  str  = ".",
    append:     bool = False,
    timestamp:  bool = True,
) -> str:
    """
    Writes a credential to a flat file with no extension.

    Args:
        credential: The credential string to save.
        filename:   Base name of the output file (no extension).
                    Defaults to 'credential'.
        directory:  Directory to write into. Created if it does not exist.
                    Defaults to current working directory.
        append:     If True,  add to existing file (one credential per line).
                    If False, overwrite file each time.
        timestamp:  If True, prefix each line with an ISO-8601 timestamp.

    Returns:
        The absolute path of the written file.

    Raises:
        ValueError: if filename contains path separators or an extension.
        OSError:    if the file cannot be written.

    Example output file — 'credential':
        2026-05-19T14:32:01  Bxkrp-Yjmt-Tqwni
        2026-05-19T14:32:08  Mtrqk-Ybp7-Eoshf
    """
    # ── Guard: no extensions, no path separators in filename ─────────────────
    if os.sep in filename or (os.altsep and os.altsep in filename):
        raise ValueError(
            f"filename must be a bare name, not a path. Got '{filename}'."
        )
    if "." in filename:
        raise ValueError(
            f"filename must have no extension. Got '{filename}'."
        )

    os.makedirs(directory, exist_ok=True)
    filepath = os.path.join(directory, filename)
    mode     = "a" if append else "w"

    line = (
        f"{datetime.now().strftime('%Y-%m-%dT%H:%M:%S')} {credential}\n"
        if timestamp
        else f"{credential}\n"
    )

    with open(filepath, mode, encoding="utf-8") as fh:
        fh.write(line)

    return os.path.abspath(filepath)

    # os.makedirs(directory, exist_ok=True)
    # filepath = os.path.join(directory, f"{filename}.txt")
    # mode     = "a" if append else "w"
    # prefix   = f"{datetime.now().isoformat()}  " if timestamp else ""  # 2 spaces to match your format

    # with open(filepath, mode) as f:
    #     f.write(f"{prefix}{credential}\n")

    # return filepath

# def load_credential(
#     filename:  str  = "credential",
#     directory: str  = ".",
#     index:     int  = -1,
#     timestamp: bool = True,
# ) -> "EncryptedCredential":
#     filepath = os.path.join(directory, filename)

#     if not os.path.exists(filepath):
#         raise FileNotFoundError(f"No credential file found at: {filepath}")

#     with open(filepath, encoding="utf-8") as f:
#         lines = [l.strip() for l in f if l.strip()]

#     if not lines:
#         raise ValueError("Credential file is empty.")

#     line = lines[index]

#     if timestamp:
#         # format is: "2026-05-19T21:37:23 <credential>"  ← single space
#         line = line.split(" ", 1)[-1].strip()

#     return EncryptedCredential(line)

# 0th node - file must be created prior to inserting credential
if __name__ == "__main__":
    cred = ""

    saved_path = save_credential(
        credential = cred,
        filename   = "credential", # input
        directory  = ".",         # Directory Structure
        append     = True,       # accumulate credentials across runs
        timestamp  = True,       # prefix each line with date-time
    )

    print(f"Saved to  : {saved_path}")

Saved to  : /workspace/Key-Storage/Module-Developer/credential


In [28]:

# initiate file save with empty credential 
# @node[0] timestamp : begin time validation 
# @node[1] timestamp : credential - apply encryption begin time validation 


# 2026-05-19T21:13:28  


In [29]:
# # from cryptography.fernet import Fernet
# import hashlib, secrets, base64, os
# import base64, hashlib, os

# class Credential(str):
#     """
#     A str subclass that adds revolving symmetric encryption.
#     Each call to .encrypt() produces a DIFFERENT ciphertext (via a fresh salt),
#     but .decrypt() always recovers the original value.
#     """

#     _MASTER_KEY = Fernet.generate_key()  # rotate this per-session or load from vault

#     def encrypt(self) -> "EncryptedCredential":
#         salt = os.urandom(16)                          # fresh salt → different output every time
#         key  = self._derive_key(salt)
#         token = Fernet(key).encrypt(self.encode())
#         payload = base64.urlsafe_b64encode(salt) + b"." + token
#         return EncryptedCredential(payload.decode(), original_type=type(self))

#     @classmethod
#     def _derive_key(cls, salt: bytes) -> bytes:
#         return base64.urlsafe_b64encode(
#             hashlib.scrypt(cls._MASTER_KEY, salt=salt, n=2**14, r=8, p=1, dklen=32)
#         )


# class EncryptedCredential(str):
#     """Returned by Credential.encrypt(). Carries .decrypt() back."""

#     def __new__(cls, value, original_type=None):
#         obj = super().__new__(cls, value)
#         obj._original_type = original_type or Credential
#         return obj

#     def decrypt(self) -> Credential:
#         raw        = base64.urlsafe_b64decode(self.split(".")[0])  # salt
#         token      = self.split(".", 1)[1].encode()                # Fernet token
#         key        = Credential._derive_key(raw)
#         plaintext  = Fernet(key).decrypt(token).decode()
#         return Credential(plaintext)

#     def encrypt(self) -> "EncryptedCredential":
#         """Decrypt first, then re-encrypt — produces a new revolving ciphertext."""
#         return self.decrypt().encrypt()

In [30]:


class Credential(str):
    """
    Zero-dependency revolving symmetric encryption.
    Uses XOR cipher with scrypt-derived key + fresh salt each round.
    """

    _MASTER_KEY = secrets.token_bytes(32)

    def encrypt(self) -> "EncryptedCredential":
        salt       = secrets.token_bytes(16)
        key        = self._derive_key(salt)
        ciphertext = self._xor(self.encode(), key)
        # pad to multiple of 3 bytes before encoding → no leftover chars
        payload    = salt + ciphertext
        padding    = (3 - len(payload) % 3) % 3
        padded     = payload + b'\x00' * padding
        encoded    = base64.urlsafe_b64encode(padded).decode()
        # store padding length so we can strip it on decrypt
        return EncryptedCredential(f"{encoded}:{padding}")

    def decrypt(self) -> "Credential":
        return self  # already plain

    @classmethod
    def _derive_key(cls, salt: bytes) -> bytes:
        return hashlib.scrypt(cls._MASTER_KEY, salt=salt, n=2**14, r=8, p=1, dklen=64)

    @staticmethod
    def _xor(data: bytes, key: bytes) -> bytes:
        return bytes(b ^ key[i % len(key)] for i, b in enumerate(data))


class EncryptedCredential(str):

    def decrypt(self) -> Credential:
        # split off padding indicator
        encoded, padding = self.rsplit(":", 1)
        padding  = int(padding)
        raw      = base64.urlsafe_b64decode(encoded.encode())
        # strip null padding bytes
        if padding:
            raw = raw[:-padding]
        salt       = raw[:16]
        ciphertext = raw[16:]
        key        = Credential._derive_key(salt)
        plaintext  = Credential._xor(ciphertext, key).decode()
        return Credential(plaintext)

    def encrypt(self) -> "EncryptedCredential":
        """Decrypt then re-encrypt — produces a new revolving ciphertext."""
        return self.decrypt().encrypt()

In [31]:

class Credential(str):
    """
    Zero-dependency revolving symmetric encryption.
    Uses XOR cipher with scrypt-derived key + fresh salt each round.
    Each .encrypt() call produces a unique ciphertext.
    """

    _MASTER_KEY = secrets.token_bytes(32)  # session key — store/load this for persistence

    def encrypt(self) -> "EncryptedCredential":
        salt       = secrets.token_bytes(16)          # fresh salt → different output every time
        key        = self._derive_key(salt)
        ciphertext = self._xor(self.encode(), key)
        payload    = base64.urlsafe_b64encode(salt + ciphertext)
        return EncryptedCredential(payload.decode())

    def decrypt(self) -> "Credential":
        return self  # already plain

    @classmethod
    def _derive_key(cls, salt: bytes) -> bytes:
        return hashlib.scrypt(cls._MASTER_KEY, salt=salt, n=2**14, r=8, p=1, dklen=64)

    @staticmethod
    def _xor(data: bytes, key: bytes) -> bytes:
        return bytes(b ^ key[i % len(key)] for i, b in enumerate(data))


class EncryptedCredential(str):

    def decrypt(self) -> Credential:
        raw        = base64.urlsafe_b64decode(self.encode())
        salt       = raw[:16]
        ciphertext = raw[16:]
        key        = Credential._derive_key(salt)
        plaintext  = Credential._xor(ciphertext, key).decode()
        return Credential(plaintext)

    def encrypt(self) -> "EncryptedCredential":
        """Re-encrypts with a new salt — revolves to a fresh ciphertext."""
        return self.decrypt().encrypt()

In [32]:

ROOT = pathlib.Path("..").resolve()          # always points to ../
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from credential       import (Credential, EncryptedCredential,
                               create_credential, format_credential,
                               save_credential, load_credential)
from ledger           import Ledger
from credential_chain import CredentialChain  # once you add it

ModuleNotFoundError: No module named 'credential'

In [33]:
if __name__ == "__main__":
    cred = Credential(create_credential(
        include_special_char=True,
        formatter=format_credential(
            group_sizes    = (8, 3, 5),
            delimiter      = ":",
            upper_count    = 1,
            digit_count    = 0,
            symbol_count   = 0,
            upper_position = "first",
        )
    ))

    print(f"Original     : {cred}")

    # enc1 = cred.encrypt()
    # print(f"Encrypted 1  : {enc1}")

    # enc2 = cred.encrypt()
    # print(f"Encrypted 2  : {enc2}")       # different from enc1 ✓

    # dec1 = enc1.decrypt()
    # print(f"Decrypted 1  : {dec1}")       # matches original ✓

    # re_enc = enc1.encrypt()
    # print(f"Re-encrypted : {re_enc}")     # revolves to new ciphertext ✓
    # print(f"Re-decrypted : {re_enc.decrypt()}")

    # ── Save ──────────────────────────────────────────────────────────────
    cred_enc = cred.encrypt()          # <-- must be encrypted, not plain cred
    saved_path = save_credential(
        credential = cred_enc,         # EncryptedCredential, not Credential
        filename   = "credential",
        directory  = ".",
        append     = True,
        timestamp  = True,
    )
    print(f"Saved to     : {saved_path}")
    print(f"Saved value  : {cred_enc}")   # verify it's the long base64 string

    # ── Load & recover ────────────────────────────────────────────────────
    loaded    = load_credential(filename="credential", directory=".", index=-1)
    recovered = loaded.decrypt()
    print(f"Loaded enc   : {loaded}")
    print(f"Recovered    : {recovered}")   # matches original ✓
    
    

Original     : Cpneewpf:Jbm:Jaayh
Saved to     : /workspace/Key-Storage/Module-Developer/credential
Saved value  : -y8SaDz6xPecYnpJ_qEQHsLd3qwLnTsjN7xkmY8cTdVXJA==
Loaded enc   : -y8SaDz6xPecYnpJ_qEQHsLd3qwLnTsjN7xkmY8cTdVXJA==
Recovered    : Cpneewpf:Jbm:Jaayh


In [34]:
# Original     : Ed:Fn:Kg
# Encrypted 1  : JbgkoJy5UVgAHlo9Xlt7oDBaNDawdE4Y
# Encrypted 2  : luk1tD8itjwc5SmO25Eg3ZCX5JycPoUy
# Decrypted 1  : Ed:Fn:Kg
# Re-encrypted : 0suGPyfPxqcWYMzf6pyNXamkQQ4X80OE
# Re-decrypted : Ed:Fn:Kg
# Saved to     : /workspace/Key-Storage/Module-Developer/credential
# Saved value  : 8hcS6J3MY3_5ODHSjInj9UhIiGAhuvbP
# Loaded enc   : 8hcS6J3MY3_5ODHSjInj9UhIiGAhuvbP
# Recovered    : Ed:Fn:Kg

In [35]:
%%writefile ../credential.py
# paste the contents of your Credential class here

Writing ../credential.py


In [36]:
import sys, pathlib
import hashlib, secrets, base64, os
import hashlib, secrets, base64, os
import random
import string
import re
import os
from datetime import datetime
import random
import string
import re

In [42]:
"""
credential_chain.py
====================
Ties the Ledger blockchain to the credential file lifecycle.

Node sequence
─────────────
  Node 0  (genesis)  – hash of the empty credential file
  Node 1             – hash of file after datetime header is written
  Node 2             – hash of file after first encrypted credential is appended
  Node N             – hash of file after every subsequent access / append

All your existing classes (Credential, EncryptedCredential, Ledger,
create_credential, format_credential, save_credential, load_credential)
are imported from wherever they live in your module tree.
Adjust the import paths to match your project layout.
"""

import hashlib
import logging
from datetime import datetime, timezone
from pathlib import Path

# ── adjust these imports to your actual module paths ───────────────────────────
# from module_developer.credential import (
#     Credential,
#     EncryptedCredential,
#     create_credential,
#     format_credential,
#     save_credential,
#     load_credential,
# )
# from module_developer.ledger import Ledger          # your blockchain class
# ──────────────────────────────────────────────────────────────────────────────

log = logging.getLogger(__name__)
log.setLevel(logging.INFO)
if not log.handlers:
    _h = logging.StreamHandler()
    _h.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
    log.addHandler(_h)


# ── helpers ────────────────────────────────────────────────────────────────────

def _hash_file(path: Path) -> str:
    """SHA-256 of current file contents (empty file → hash of b'')."""
    data = path.read_bytes() if path.exists() else b""
    return hashlib.sha256(data).hexdigest()


def _now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


# ── core lifecycle ─────────────────────────────────────────────────────────────

class CredentialChain:
    """
    Manages a credential file whose every mutation is recorded as a
    blockchain node in a paired Ledger.

    Parameters
    ----------
    cred_path   : Path to the credential flat-file  (e.g. Path("credential"))
    ledger_path : Path to the ledger JSON file       (e.g. Path("credential.ledger"))
    directory   : directory that holds both files    (default: current dir)
    """

    def __init__(
        self,
        cred_filename:   str  = "credential",
        ledger_filename: str  = "credential.ledger",
        directory:       str  = ".",
    ):
        self.dir          = Path(directory)
        self.cred_path    = self.dir / cred_filename
        self.ledger_path  = self.dir / ledger_filename

    # ── public API ─────────────────────────────────────────────────────────────

    # def initialise(self) -> Ledger:
    #     """
    #     Create a brand-new credential file + ledger from scratch.

    #     Step 1 – touch empty file  → Node 0  (genesis, hash of empty file)
    #     Step 2 – write datetime    → Node 1  (hash of file with header)

    #     Returns the open Ledger.
    #     """
    #     # ── guard: refuse to overwrite an existing chain ───────────────────
    #     if self.ledger_path.exists():
    #         raise FileExistsError(
    #             f"Ledger already exists at {self.ledger_path}. "
    #             "Call .access() to append, or delete both files to start fresh."
    #         )

    #     # ── STEP 1: empty file → genesis node ─────────────────────────────
    #     self.cred_path.write_bytes(b"")                          # touch
    #     empty_hash = _hash_file(self.cred_path)

    #     ledger = Ledger(self.ledger_path)                        # creates genesis block internally
    #     # Override genesis key_hash so it reflects the actual file state.
    #     # If your Ledger.append() is the only public mutator, add the hash
    #     # as the first real append instead (see comment below).
    #     #
    #     # Option A – if Ledger exposes the genesis block for editing:
    #     ledger.chain[0]["key_hash"] = empty_hash
        
    #     ledger._save()          # persist the updated genesis block
    #     #
    #     # Option B – if genesis is immutable, uncomment this and remove A:
    #     # ledger.append(empty_hash, b"")   # Node 0: empty-file snapshot

    #     log.info("Node 0 (genesis) – empty file hash: %s", empty_hash[:16])

    #     # ── STEP 2: write datetime header → Node 1 ────────────────────────
    #     timestamp = _now_iso()
    #     self.cred_path.write_text(timestamp + "\n", encoding="utf-8")
    #     ts_hash = _hash_file(self.cred_path)
    #     ts_bytes = timestamp.encode()

    #     ledger.append(ts_hash, ts_bytes)
    #     log.info("Node 1 (datetime) – file hash: %s", ts_hash[:16])

    #     return ledger
    def initialise(self) -> Ledger:
        if self.ledger_path.exists():
            raise FileExistsError(
                f"Ledger already exists at {self.ledger_path}. "
                "Call .access() to append, or delete both files to start fresh."
            )
    
        # ── Node 0: empty file → genesis is created automatically by Ledger.__init__
        self.cred_path.write_bytes(b"")
        ledger = Ledger(self.ledger_path)          # genesis block made here internally
        log.info("Genesis block created.")
    
        # ── Node 1: record the empty file state as first real block
        empty_hash = _hash_file(self.cred_path)
        ledger.append(empty_hash, b"")
        log.info("Block 1 — empty file hash: %s", empty_hash[:16])
    
        # ── Node 2: write datetime, record that state
        timestamp = _now_iso()
        self.cred_path.write_text(timestamp + "\n", encoding="utf-8")
        ts_hash = _hash_file(self.cred_path)
        ledger.append(ts_hash, timestamp.encode())
        log.info("Block 2 — datetime written: %s", timestamp)
    
        return ledger

    def add_credential(
        self,
        ledger: Ledger,
        *,
        include_special_char: bool = True,
        group_sizes:    tuple = (8, 3, 5),
        delimiter:      str   = ":",
        upper_count:    int   = 1,
        digit_count:    int   = 0,
        symbol_count:   int   = 0,
        upper_position: str   = "first",
    ) -> tuple[Credential, "EncryptedCredential", Path]:
        """
        Generate, encrypt, and append one credential to the file.
        Adds Node N to the ledger (hash of file *after* the append).

        Returns (plain_credential, encrypted_credential, saved_path)
        """
        # validate the chain before touching the file
        if not ledger.is_valid():
            raise RuntimeError("Ledger integrity check failed before credential write.")

        # ── build credential ───────────────────────────────────────────────
        cred = Credential(
            create_credential(
                include_special_char=include_special_char,
                formatter=format_credential(
                    group_sizes    = group_sizes,
                    delimiter      = delimiter,
                    upper_count    = upper_count,
                    digit_count    = digit_count,
                    symbol_count   = symbol_count,
                    upper_position = upper_position,
                ),
            )
        )
        cred_enc = cred.encrypt()

        # ── append to file ─────────────────────────────────────────────────
        saved_path = save_credential(
            credential = cred_enc,
            filename   = self.cred_path.name,
            directory  = str(self.dir),
            append     = True,
            timestamp  = True,
        )

        # ── record new file state as a ledger node ─────────────────────────
        file_hash  = _hash_file(self.cred_path)
        enc_bytes  = str(cred_enc).encode()
        ledger.append(file_hash, enc_bytes)
        node_index = len(ledger.chain) - 1
        log.info("Node %d – credential appended, file hash: %s", node_index, file_hash[:16])

        return cred, cred_enc, saved_path

    def access(self) -> tuple[Ledger, "EncryptedCredential"]:
        """
        Open an existing credential chain, validate it, load the latest
        credential, and record an access node.

        Returns (ledger, latest_encrypted_credential)
        """
        if not self.ledger_path.exists():
            raise FileNotFoundError(
                f"No ledger at {self.ledger_path}. Call .initialise() first."
            )

        ledger = Ledger(self.ledger_path)

        if not ledger.is_valid():
            raise RuntimeError(
                "Ledger integrity check FAILED on load – file may have been tampered with."
            )

        log.info("Chain valid. %d nodes loaded.", len(ledger.chain))

        # ── load most-recent credential from file ──────────────────────────
        loaded    = load_credential(
            filename  = self.cred_path.name,
            directory = str(self.dir),
            index     = -1,
        )
        recovered = loaded.decrypt()
        log.info("Loaded credential (decrypted): %s", recovered)

        # ── record this access as a new node ───────────────────────────────
        file_hash = _hash_file(self.cred_path)
        access_ts = _now_iso().encode()
        ledger.append(file_hash, access_ts)
        node_index = len(ledger.chain) - 1
        log.info("Node %d – access recorded, file hash: %s", node_index, file_hash[:16])

        return ledger, loaded

    def verify(self) -> bool:
        """Quick integrity check without side-effects."""
        if not self.ledger_path.exists():
            log.warning("No ledger file found.")
            return False
        ledger = Ledger(self.ledger_path)
        valid  = ledger.is_valid()
        log.info("Chain valid: %s (%d nodes)", valid, len(ledger.chain))
        return valid


# ── entry point (mirrors your __main__ block) ──────────────────────────────────

if __name__ == "__main__":

    chain = CredentialChain(
        cred_filename   = "credential",
        ledger_filename = "credential.ledger",
        directory       = "../module-developer/",
    )

    # ── first run: build the chain from scratch ────────────────────────────
    ledger = chain.initialise()
    print(f"After init  – chain length : {len(ledger.chain)}")   # expect 2 (nodes 0 & 1)

    # ── add the first credential (Node 2) ─────────────────────────────────
    cred, cred_enc, saved_path = chain.add_credential(ledger)
    print(f"Original    : {cred}")
    print(f"Encrypted   : {cred_enc}")
    print(f"Saved to    : {saved_path}")
    print(f"After write – chain length : {len(ledger.chain)}")   # expect 3

    # ── simulate a later access (Node 3) ──────────────────────────────────
    ledger2, latest_enc = chain.access()
    recovered = latest_enc.decrypt()
    print(f"Recovered   : {recovered}")
    print(f"After access – chain length : {len(ledger2.chain)}")  # expect 4

    # ── integrity check ────────────────────────────────────────────────────
    assert chain.verify(), "Chain should be valid!"
    print("All checks passed ✔")

INFO: Genesis block created.
INFO: Block 1 — empty file hash: e3b0c44298fc1c14
INFO: Block 2 — datetime written: 2026-05-20T03:28:44.582431+00:00
INFO: Node 3 – credential appended, file hash: ad603a7d4b2fcb3b
INFO: Chain valid. 4 nodes loaded.
INFO: Loaded credential (decrypted): Nbgolhcp:Vyk:Hdywb
INFO: Node 4 – access recorded, file hash: ad603a7d4b2fcb3b
INFO: Chain valid: True (5 nodes)


After init  – chain length : 3
Original    : Nbgolhcp:Vyk:Hdywb
Encrypted   : EsEkfHb-zIKdeCFB3lZq5OuDveubgG-NtjuGLN0GqbU0ew==
Saved to    : /workspace/Key-Storage/module-developer/credential
After write – chain length : 4
Recovered   : Nbgolhcp:Vyk:Hdywb
After access – chain length : 5
All checks passed ✔
